A simple code for parsing images and detecting objects with CLIP

## Parsing

In [1]:
import os
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from pathlib import Path
import shutil

In [2]:
url = "https://en.wikipedia.org/wiki/Orange_(fruit)"
# url = "https://en.wikipedia.org/wiki/Machine_learning"

In [3]:
os.makedirs("images", exist_ok=True)

In [4]:
r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'})
r.raise_for_status()

In [5]:
soup = BeautifulSoup(r.text, "html.parser")

In [6]:
img_tags = soup.find_all('img')
print(img_tags[:5])

[<img alt="" aria-hidden="true" class="mw-logo-icon" height="50" src="/static/images/icons/enwiki-25.svg" width="50"/>, <img alt="Wikipedia" class="mw-logo-wordmark" src="/static/images/mobile/copyright/wikipedia-wordmark-en-25.svg" style="width: 8.75em; height: 1.375em;"/>, <img alt="The Free Encyclopedia" class="mw-logo-tagline" height="11" src="/static/images/mobile/copyright/wikipedia-tagline-en-25.svg" style="width: 8.75em; height: 0.6875em;" width="140"/>, <img alt="This is a good article. Click here for more information." class="mw-file-element" data-file-height="185" data-file-type="drawing" data-file-width="180" decoding="async" height="20" id="mwCSc" loading="lazy" resource="https://en.wikipedia.org/wiki/File:Symbol_support_vote.svg" src="//thumb.wikimedia.org/wikipedia/en/thumb/9/94/Symbol_support_vote.svg/20px-Symbol_support_vote.svg.png?utm_source=en.wikipedia.org&amp;utm_campaign=parser&amp;utm_content=thumbnail" srcset="//thumb.wikimedia.org/wikipedia/en/thumb/9/94/Symbo

In [7]:
img_tags[0]

<img alt="" aria-hidden="true" class="mw-logo-icon" height="50" src="/static/images/icons/enwiki-25.svg" width="50"/>

In [14]:
for index, img in enumerate(img_tags):

    img_url = img.get("src")
    if not img_url:
        continue
    # img_url = urljoin(url, img_url)

    # fixing wiki paths
    img_url = img_url.strip()
    if img_url.startswith("//"):
        img_url = f"https:{img_url}"
    else:
        img_url = urljoin(url, img_url)

    img_url = img_url.split("&utm")[0]
    img_url = img_url.split("?utm")[0]

    print(img_url)

    try:
        img_data = requests.get(img_url, headers={'User-Agent': 'Mozilla/5.0'}
                                ).content
        file_ext = os.path.splitext(img_url)[1]
        file_name = os.path.join('images', f'image_{index}{file_ext}')

        with open(file_name, 'wb') as handler:
            handler.write(img_data)

    except:
        print("can't load")

https://en.wikipedia.org/static/images/icons/enwiki-25.svg
https://en.wikipedia.org/static/images/mobile/copyright/wikipedia-wordmark-en-25.svg
https://en.wikipedia.org/static/images/mobile/copyright/wikipedia-tagline-en-25.svg
https://thumb.wikimedia.org/wikipedia/en/thumb/9/94/Symbol_support_vote.svg/20px-Symbol_support_vote.svg.png
https://thumb.wikimedia.org/wikipedia/en/thumb/1/1b/Semi-protection-shackle.svg/20px-Semi-protection-shackle.svg.png
https://thumb.wikimedia.org/wikipedia/commons/thumb/e/e3/Oranges_-_whole-halved-segment.jpg/500px-Oranges_-_whole-halved-segment.jpg
https://thumb.wikimedia.org/wikipedia/commons/thumb/b/bd/Orange_Blossom.JPG/330px-Orange_Blossom.JPG
https://thumb.wikimedia.org/wikipedia/commons/thumb/9/9e/Orange_tree_fruiting.jpg/250px-Orange_tree_fruiting.jpg
https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b0/OrangeBloss_wb.jpg/330px-OrangeBloss_wb.jpg
https://thumb.wikimedia.org/wikipedia/commons/thumb/6/6e/Laranxeira_Naranjo_GFDL.JPG/330px-Laranxe

In [9]:
# clearing the images folder

folder_path = Path("images")

for item in folder_path.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()

## ML

In [10]:
import torch
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

...

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [15]:
# image_path = "/content/images/image_24.jpg"
image_path = "/content/images/image_5.jpg"
target_object = "fruits"

In [16]:
image = Image.open(image_path)

In [17]:
text_queries = [f"a photo of {target_object}", "a photo of something else"]
inputs = processor(text=text_queries, images=image, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

In [18]:
with torch.no_grad():
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=-1).cpu()

object_prob = probs[0][0].item()

print(f"Probability having '{target_object}': {object_prob:.2%}")
if object_prob > 0.7:
    print("Object detected!")
else:
    print("No object...")

Probability having 'fruits': 97.69%
Object detected!


# checking all files

In [19]:
valid_extensions = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
folder_path = Path("images")
all_files = [f for f in os.listdir(folder_path) if f.lower().endswith(valid_extensions)]

In [20]:
print(f"Found images: {len(all_files)}")

Found images: 47


In [22]:
found_images = []

for file_name in all_files:
    image_path = os.path.join(folder_path, file_name)

    try:
        image = Image.open(image_path).convert("RGB")

        inputs = processor(text=text_queries, images=image, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=-1).cpu()

        # object_prob = probs.item()
        object_prob = probs[0][0].item()
        if object_prob >= 0.7:
            found_images.append((file_name, object_prob))

    except Exception as e:
        print(f"Error on {file_name}: {e}")

print("\n=== Results ===")
if found_images:
    print(f"Object '{target_object}' found in following files:")
    for file_name, prob in found_images:
        print(f" - {file_name} (Probability: {prob:.2%})")
else:
    print(f"No '{target_object}' found.")


=== Results ===
Object 'fruits' found in following files:
 - image_39.jpg (Probability: 91.37%)
 - image_18.jpg (Probability: 73.60%)
 - image_8.jpg (Probability: 95.26%)
 - image_9.JPG (Probability: 95.61%)
 - image_22.jpg (Probability: 99.41%)
 - image_11.png (Probability: 97.04%)
 - image_31.jpg (Probability: 96.07%)
 - image_21.jpg (Probability: 99.00%)
 - image_10.png (Probability: 99.83%)
 - image_15.JPG (Probability: 99.21%)
 - image_20.jpg (Probability: 99.03%)
 - image_5.jpg (Probability: 97.69%)
 - image_32.jpg (Probability: 81.23%)
 - image_17.jpg (Probability: 98.81%)
 - image_16.JPG (Probability: 97.19%)
 - image_6.JPG (Probability: 86.95%)
 - image_19.jpg (Probability: 92.45%)
 - image_37.jpg (Probability: 97.38%)
 - image_40.jpg (Probability: 99.67%)
